# 02 · Load and clean

**What this notebook does:** loads the three files from notebook 01, cleans them, and saves them as tables.
On Databricks the tables go to Unity Catalog (`workspace.complaint_radar`); locally they go to a DuckDB file.
Every later notebook reads these tables, never the raw files.

### Connect
`setup()` creates the schema on Databricks, or opens the local DuckDB file.

In [1]:
from radar import setup, save_table, run_sql, DATA
setup()

'Local: tables go to data/complaint_radar.duckdb'

### Load the sample of complaints with narratives

In [2]:
import pandas as pd
data = pd.read_parquet(DATA / "complaints_sample.parquet")
print(f"{len(data):,} complaints, {data['date_received'].min():%d %b %Y} to {data['date_received'].max():%d %b %Y}")
data.head(3)

27,463 complaints, 01 Jan 2025 to 29 Jul 2026


,date_received,product,sub_product,issue,sub_issue,narrative,company,state,submitted_via,company_response,timely,complaint_id,month
0,2025-01-01,"Payday loan, title loan, personal loan, or adv...",Installment loan,Charged fees or interest you didn't expect,NaN,American First Finance ( AFF ) is a predatory ...,"American First Finance, Inc.",FL,Web,Closed with monetary relief,Yes,11345770,2025-01-01
1,2025-01-01,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Trouble with how payments are being handled,Aidvantage operates under horrible business pr...,"Maximus Federal Services, Inc.",CA,Web,Closed with explanation,Yes,11345877,2025-01-01
2,2025-01-01,Vehicle loan or lease,Loan,Managing the loan or lease,Problem with fees charged,I have attached my last XXXX monthly statement...,"SANTANDER HOLDINGS USA, INC.",IL,Web,Closed with explanation,Yes,11343507,2025-01-01


### Shorter product names
The CFPB's product names are long ("Payday loan, title loan, personal loan, or advance loan"). Charts and model
labels are easier to read with short names. Anything not in the list keeps its original name.

In [3]:
SHORT = {
    "Credit reporting or other personal consumer reports": "Credit reporting",
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
    "Debt collection": "Debt collection",
    "Checking or savings account": "Bank account",
    "Credit card": "Credit card",
    "Credit card or prepaid card": "Credit card",
    "Prepaid card": "Prepaid card",
    "Money transfer, virtual currency, or money service": "Money transfer",
    "Mortgage": "Mortgage",
    "Vehicle loan or lease": "Vehicle loan",
    "Student loan": "Student loan",
    "Payday loan, title loan, personal loan, or advance loan": "Personal loan",
    "Payday loan, title loan, or personal loan": "Personal loan",
    "Debt or credit management": "Debt management",
}
data["product_full"] = data["product"]                               # keep the original name too
data["product"] = data["product"].map(SHORT).fillna(data["product"])
data["product"].value_counts()

product
Credit reporting    4000
Credit card         4000
Bank account        4000
Money transfer      4000
Debt collection     4000
Mortgage            2201
Vehicle loan        1736
Student loan        1476
Personal loan       1179
Prepaid card         522
Debt management      349
Name: count, dtype: int64

### Clean the text for the models
The CFPB hides personal details before publishing: names become `XXXX`, dates become `XX/XX/2025`, amounts
become `{$500.00}`. Those masks carry no meaning, so we remove them from the copy the models read.
The original narrative stays untouched for display.

In [4]:
import re

def clean(text: str) -> str:
    text = re.sub(r"X{2,}", " ", text)                    # XXXX masks
    text = re.sub(r"\{\$[\d,\.]+\}", " amount ", text)     # {$500.00} -> "amount"
    text = re.sub(r"[^A-Za-z' ]+", " ", text)              # drop numbers and punctuation
    return re.sub(r"\s+", " ", text).strip().lower()

data["text"] = data["narrative"].fillna("").map(clean)
data["words"] = data["text"].str.split().str.len()         # length of each complaint in words
data["words"].describe().round(0)

count    27463.0
mean       185.0
std        200.0
min          0.0
25%         75.0
50%        126.0
75%        234.0
max       4516.0
Name: words, dtype: float64

### Drop complaints too short to learn from
A handful of narratives are just a few words ("see attached"). Under 20 words there is nothing to classify.

In [5]:
before = len(data)
data = data[data["words"] >= 20].reset_index(drop=True)
print(f"dropped {before - len(data):,} very short complaints, {len(data):,} left")

dropped 905 very short complaints, 26,558 left


### Save the complaints table

In [6]:
data["date_received"] = pd.to_datetime(data["date_received"])
save_table(data, "complaints")

'complaints: 26,558 rows'

### Load and save the counts over all complaints
These include complaints without a narrative, so the trends reflect the whole database, not just our sample.

In [7]:
data = pd.read_parquet(DATA / "complaints_monthly.parquet")
data["product"] = data["product"].map(SHORT).fillna(data["product"])
data = data.groupby(["month", "product", "issue", "has_narrative"], dropna=False)["complaints"].sum().reset_index()
save_table(data, "monthly_counts")

'monthly_counts: 4,820 rows'

In [8]:
data = pd.read_parquet(DATA / "companies.parquet")
data["product"] = data["product"].map(SHORT).fillna(data["product"])
data = data.groupby(["company", "product", "company_response", "timely"], dropna=False)["complaints"].sum().reset_index()
save_table(data, "company_counts")

'company_counts: 16,442 rows'

### Check the tables
A quick count from each table, using SQL, to make sure everything landed.

In [9]:
data = run_sql("""
    SELECT 'complaints' AS table_name, COUNT(*) AS rows_ FROM complaints
    UNION ALL SELECT 'monthly_counts', COUNT(*) FROM monthly_counts
    UNION ALL SELECT 'company_counts', COUNT(*) FROM company_counts
""")
data

,table_name,rows_
0,complaints,26558
1,monthly_counts,4820
2,company_counts,16442
